In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Reading the file 

In [0]:
df = spark.read.format("csv")\
             .option("header", True)\
             .option("inferSchema", True)\
             .load("/Workspace/Users/edwinvictor73@gmail.com/Data-science/raw/ipl_ball_by_ball.csv")

In [0]:
df.limit(10).display()

### Cleaning the column season before filtering as the value in season column is in two different form

In [0]:
df_season_col_cleaned = df.withColumn("season", \
    when(col("season").rlike(r"^\d{4}/\d{2}$"),year(col("date")))\
        .otherwise(col("season")))\
        .withColumn("season", col("season").cast(IntegerType()))\
        .filter(col("season") >= 2020)
        
                        
    

In [0]:
df_season_col_cleaned.limit(10).display()


### Creating new columns for batsmen metrics 

In [0]:
df_season_col_cleaned.limit(5).display()

In [0]:
df_total_runs = df_season_col_cleaned.groupby(col("batter"))\
    .agg(sum("batter_runs").alias("total_runs"),\
     sum(when(col("is_powerplay") == 1, col("batter_runs"))
         .otherwise(0)
         ).alias("Runs_scored_in_powerplay"),\
     sum(when(col("is_middle_overs") == 1, col("batter_runs"))
         .otherwise(0)
        ).alias("Runs_scored_in_middle_overs"),\
     sum(when(col("is_death_overs") == 1, col("batter_runs"))
         .otherwise(0)
         ).alias("Runs_scored_in_death_overs")
     )

In [0]:
df_total_runs.sort("Runs_scored_in_death_overs", ascending=False).limit(20).display()